In [ ]:
# Downloading Data - Set of 50 videos from each category
!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category real

!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category fake

!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category Face2Face

!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category FaceSwap

!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category NeuralTextures

!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category DFD_real

!bash download_ff_data.sh --num_videos 50 --frame_skip 20 --category DFD_fake

In [ ]:
# Downloading Data - Larger/Balanced Dataset
!bash download_ff_data.sh --num_videos 200 --frame_skip 20 --category real

!bash download_ff_data.sh --num_videos 80 --frame_skip 20 --category fake

!bash download_ff_data.sh --num_videos 80 --frame_skip 20 --category Face2Face

!bash download_ff_data.sh --num_videos 80 --frame_skip 20 --category FaceSwap

!bash download_ff_data.sh --num_videos 80 --frame_skip 20 --category NeuralTextures

!bash download_ff_data.sh --num_videos 200 --frame_skip 20 --category DFD_real

!bash download_ff_data.sh --num_videos 80 --frame_skip 20 --category DFD_fake



In [ ]:
'''
group videos by name to get into format of map
'''

from collections import defaultdict
import re
import os
import pandas as pd

"""
Prereqs: download 10 vids/directory in all_dirs via download_ff_data.sh
"""
def group_by_video(directory):
    groups = defaultdict(list)
    for video_name in os.listdir(directory):
        video_path = os.path.join(directory, video_name)
        if not os.path.isdir(video_path):
            continue
        frames = sorted(f for f in os.listdir(video_path) if f.endswith('.png'))
        if frames:
            # change: doesn't prepend video name to frame filename
            groups[video_name] = [f for f in frames]

    result = []
    for key, frames in groups.items():
        result.append([key, frames])
    return result


def extract_original_vid(path: str):
    # use this to ensure that original videos and their manipulated derivatives
    # are all in the same side of the split, so that the model is only evaluated on data
    # it hasn't seen before

    name = Path(path).name

    # matches DFD fakes, first number is the og, then name should match
    m = re.fullmatch(r'(\d+)_\d+__(.+)__([A-Z0-9]{8})', name)
    if m:
        return f"{m.group(1)}__{m.group(2)}"

    # DFD reals
    m = re.fullmatch(r'(\d+)__(.+)', name)
    if m:
        return f"{m.group(1)}__{m.group(2)}"

    # rest of the fakes, first number is the target
    m = re.fullmatch(r'(\d+)_(\d+)', name)
    if m:
        return m.group(1)

    # FF reals
    if re.fullmatch(r'\d+', name):
        return name

    return None

In [ ]:
from numpy.random.mtrand import f
'''
create csv storing:
Video_name, frame 1, frame 2, .. frame 10
'''

all_dirs = ['/content/FF_data/real/original_sequences/youtube/c40/images',
            '/content/FF_data/fake/manipulated_sequences/Deepfakes/c40/images',
            '/content/FF_data/DFD_real/original_sequences/actors/c40/images',
            '/content/FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images',
            '/content/FF_data/Face2Face/manipulated_sequences/Face2Face/c40/images',
            '/content/FF_data/FaceSwap/manipulated_sequences/FaceSwap/c40/images',
            '/content/FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images',
            ]

# Check each directory has 50 videos
print("=== Directory Video Count Check ===")
for d in all_dirs:
    d_videos = d.replace('images', 'videos')
    if not os.path.exists(d_videos):
        print(f"MISSING DIR: {d_videos}")
        continue
    videos = [v for v in os.listdir(d_videos) if v.endswith(('.mp4', '.avi', '.mov', '.mkv'))]
    if len(videos) != 50:
        print(f"WRONG COUNT ({len(videos)}/50): {d_videos}")
print("=== Check Complete ===")

'''
0 - Authentic
1 - Deepfake
'''

def create_datamap(all_dirs, frames_per_clip=10, skip=1):
    data = {
        'video_path': [],
        "video_name": []
    }
    for i in range(10):
        data['frame_' + str(i)] = []

    data['label'] = []

    for dir in all_dirs:
        label = 1
        if 'original' in dir:
            label = 0
        # list of videos in dir, along with a list of frame filenames with them
        #[[vid, [vid/fr1.png, vid/fr2.png, ...]], ...]
        video_map = group_by_video(dir)
        for video, frames in video_map:
            i = 0
            for j in range(frames_per_clip * skip, len(frames), frames_per_clip * skip):
                if skip == 1:
                    split = frames[i:j]
                else:
                    split = [frames[idx] for idx in range(i, j, skip)]
                i = j
                data['video_path'].append(os.path.join(dir, video))
                data["video_name"].append(video)
                data['label'].append(label)
                for k in range(len(split)):
                    data['frame_' + str(k)].append(split[k])
    return data

data_map = create_datamap(all_dirs)
df = pd.DataFrame(data_map)
df['original_vid'] = df['video_name'].apply(extract_original_vid)
df.to_csv('whole_dataset_splits.csv')
